# 03 - Context Engineering: measuring what survives, not what shrinks

Notebooks 01 and 02 fixed how the harness *learns*. This notebook fixes how it
*forgets* - and, more importantly, how you find out what it forgot.

The design review's complaint about the original compaction and offloading
sections was not that they were wrong, but that they measured the wrong thing:

| Review gap (8.1 / 8.2b) | This notebook |
|---|---|
| Card judged by character count | **Fidelity probe**: is the fact from turn 4 still there at turn 40, when it is finally needed? |
| "The bytes are recoverable" declared a win | **Agent in the loop**: the model must notice the reference, fetch it, and use the payload |
| Offloading and promotion never tested together | Offloaded blobs are **barred from promotion**, and the notebook proves it |

A smaller card is trivially easy to produce - throw everything away and the chart
looks fantastic. The only interesting question is what it still knows.

## 0 · Setup

Connect, verify notebook 01's artifacts, and check in on notebook 02 without
insisting on it - this notebook needs 02's scratch table, but not its scheduler.

In [ ]:
import json
import random

import matplotlib.pyplot as plt

from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings
from cityops_harness.db import get_connection
from cityops_harness.llm import chat_model
from cityops_harness.state import require, verify
from cityops_harness.tracing import init_tracing
from cityops_harness import context, promote

settings = load_settings()
conn = get_connection(settings)
require(conn, "01_self_improving_copilot")
CHAT = chat_model(settings)
HANDLER = init_tracing(settings)
# Langfuse is optional all the way through: every invoke passes config=CFG.
CFG = {"callbacks": [HANDLER]} if HANDLER else {}
ok(f"connected - provider={settings.llm_provider}, langfuse={settings.langfuse_mode}")

Notebook 02's status is reported, not enforced. Its scheduler jobs need
`CREATE JOB`, which is a database grant, not a context-engineering concept -
failing this notebook over it would be the wrong dependency. The one thing we
genuinely need from 02 is its scratch table, so we backfill that if it is absent.

In [ ]:
for desc, passed in verify(conn, "02_scheduled_briefings"):
    print(f"  {'✓' if passed else '·'} {desc}")


def ddl(sql):
    """Idempotent DDL: ORA-00955 (name already used) is the success case."""
    with conn.cursor() as cur:
        try:
            cur.execute(sql)
        except Exception as exc:
            if "ORA-00955" not in str(exc):
                raise


ddl("""CREATE TABLE HARNESS_SCRATCH (
         PATH        VARCHAR2(400) PRIMARY KEY,
         CONTENT     CLOB,
         STATUS      VARCHAR2(1) DEFAULT 'N' NOT NULL,
         CREATED_AT  TIMESTAMP DEFAULT SYSTIMESTAMP)""")
ok("notebook 02 scratch store available (backfilled if it was missing)")

## 1 · A long inspection season

Compaction only gets interesting when the transcript outgrows the window, so we
need length. Generating 40 turns with an LLM would be slow, expensive, and
different on every run - and none of that variation would teach anything. The
season below is built deterministically from *real* findings in
`CITY_INSPECTION_FINDING`, seeded so every learner compacts the same conversation.

Three turns carry a **planted fact**: a specific, checkable detail we will come
back for at the very end, long after compaction has had every opportunity to
discard it.

In [ ]:
ddl("""CREATE TABLE HARNESS_TRANSCRIPT (
         TURN_NO          NUMBER PRIMARY KEY,
         SPEAKER          VARCHAR2(20),
         CONTENT          CLOB,
         PLANTED_FACT_ID  VARCHAR2(10))""")

with conn.cursor() as cur:
    cur.execute("DELETE FROM HARNESS_TRANSCRIPT")
    cur.execute(
        "SELECT asset_id, category, severity, DBMS_LOB.SUBSTR(description, 400, 1)"
        "  FROM CITY_INSPECTION_FINDING ORDER BY finding_id"
    )
    FINDINGS = cur.fetchall()

check(len(FINDINGS) > 20, f"{len(FINDINGS)} real findings available to narrate")

In [ ]:
# The three facts the probe will hunt for, and the turns they land on.
PLANTED = {
    4:  ("f1", "Pier 3 bearing plate is corroded through; it is the single reason "
               "Harbor Bridge was prioritised for Q3."),
    11: ("f2", "The Riverside Culvert inspection used the 2019 rating scale, so its "
               "grades are not comparable with the rest of this season."),
    19: ("f3", "Inspector Okafor signed off every North District report this season, "
               "so a single reviewer bias applies to all of them."),
}

rng = random.Random(20260722)
TURNS = []
for turn_no in range(1, 41):
    if turn_no in PLANTED:
        fact_id, text = PLANTED[turn_no]
        TURNS.append((turn_no, "inspector", text, fact_id))
        continue
    asset, category, severity, desc = rng.choice(FINDINGS)
    TURNS.append((
        turn_no,
        "inspector" if turn_no % 2 else "coordinator",
        f"Turn {turn_no}: on {asset}, a {severity.lower()} {category.lower()} issue - "
        f"{(desc or '').strip()}",
        None,
    ))

with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO HARNESS_TRANSCRIPT (turn_no, speaker, content, planted_fact_id)"
        " VALUES (:1, :2, :3, :4)",
        TURNS,
    )
conn.commit()

check(len(TURNS) == 40, "40-turn season written")
check(sum(1 for t in TURNS if t[3]) == 3, "3 planted facts, at turns 4, 11 and 19")

## 2 · Where the card lives

Every compaction writes a **new version** rather than overwriting the last one.
That is what makes the size chart and the fidelity probe possible after the fact:
you can see not just the final card, but the moment a fact fell out of it.

In [ ]:
ddl("""CREATE TABLE HARNESS_CARD (
         CARD_VERSION      NUMBER PRIMARY KEY,
         CREATED_AT        TIMESTAMP DEFAULT SYSTIMESTAMP,
         TURNS_COVERED     NUMBER,
         CARD_JSON         CLOB,
         TRANSCRIPT_CHARS  NUMBER,
         CARD_CHARS        NUMBER)""")

with conn.cursor() as cur:
    cur.execute("DELETE FROM HARNESS_CARD")
conn.commit()


def save_card(version, turns_covered, card, transcript_chars):
    """Persist one compaction version. Returns the rendered card text."""
    body = context.render_card(card)
    # ✏️ TODO(1): insert the card version, and return `body`.
    #   Columns: CARD_VERSION, TURNS_COVERED, CARD_JSON, TRANSCRIPT_CHARS, CARD_CHARS.
    #   CARD_CHARS is len(body) - the size half of the measurement.
    #   ... your code here ...
    return body


def latest_card():
    with conn.cursor() as cur:
        cur.execute(
            "SELECT card_json FROM HARNESS_CARD"
            " ORDER BY card_version DESC FETCH FIRST 1 ROWS ONLY"
        )
        row = cur.fetchone()
    return context.parse_card(row[0].read() if hasattr(row[0], "read") else row[0]) if row else None


EMPTY_CARD = {"facts": [], "decisions": [], "open_questions": []}
ok("card store ready (versioned, never overwritten)")

## 3 · Compaction

We walk the season one turn at a time. Whenever the pending transcript reaches the
budget, the model folds it into the running card and the buffer resets.

Two design choices matter more than the prompt:

1. **The card has a fixed schema** (`facts`, `decisions`, `open_questions`). A
   free-form summary cannot be probed; a structured one can be, by set membership.
2. **The merge is additive** (`context.merge_card`). The model proposes; it does
   not get to silently delete. A fact leaves the card only when the model
   explicitly corrects it - which means anything that *does* go missing is a real
   finding about compaction, not an artefact of us overwriting the card each time.

In [ ]:
COMPACTION_PROMPT = (
    "You are maintaining a compaction card for a bridge-inspection season.\n"
    "Return ONLY JSON with keys: facts, decisions, open_questions.\n"
    "  facts: list of {{id, text, turn}} - durable, specific claims. Reuse an\n"
    "    existing id to correct it; invent a new id (f4, f5, ...) otherwise.\n"
    "  decisions: list of strings. open_questions: list of strings.\n"
    "Keep anything a colleague would need in three months.\n\n"
    "CURRENT CARD:\n{card}\n\nNEW TURNS:\n{turns}"
)


def compact(card, pending_turns):
    """Fold pending turns into the card. Returns the merged card."""
    prompt = COMPACTION_PROMPT.format(
        card=context.render_card(card),
        turns="\n".join(f"[{n}] {who}: {text}" for n, who, text, _ in pending_turns),
    )
    # ✏️ TODO(2): call the model, parse its card, and merge it into `card`.
    #   Use CHAT.invoke(prompt, config=CFG), context.parse_card, context.merge_card.
    #   Strip any ```json fence before parsing.
    #   ... your code here ...
    return card

ok("compaction step defined")

In [ ]:
# The season is a scale model: ~12k characters, where a real one runs into the
# millions. Rather than inflate the transcript - slow, expensive, and no more
# instructive - we shrink the budget by the same factor. The mechanism, and
# everything it can lose, is identical.
CARD_BUDGET = 2_000

card = dict(EMPTY_CARD)
pending, transcript_chars, version = [], 0, 0

for turn in TURNS:
    pending.append(turn)
    transcript_chars += len(turn[2])
    if context.compaction_due(sum(len(t[2]) for t in pending), CARD_BUDGET):
        version += 1
        card = compact(card, pending)
        save_card(version, turn[0], card, transcript_chars)
        print(f"  v{version}: compacted through turn {turn[0]} - "
              f"{len(card['facts'])} facts, {len(context.render_card(card))} chars")
        pending = []

if pending:  # never strand the tail of the season
    version += 1
    card = compact(card, pending)
    save_card(version, TURNS[-1][0], card, transcript_chars)
    print(f"  v{version}: final fold through turn {TURNS[-1][0]}")

check(version >= 2, f"{version} card versions written across the season")

In [ ]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT card_version, transcript_chars, card_chars"
        "  FROM HARNESS_CARD ORDER BY card_version"
    )
    rows = cur.fetchall()

versions = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(versions, [r[1] for r in rows], marker="o", label="transcript (cumulative)")
ax.plot(versions, [r[2] for r in rows], marker="o", label="compaction card")
ax.set_yscale("log")
ax.set_xlabel("card version")
ax.set_ylabel("characters (log scale)")
ax.set_title("What compaction saves")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

ratio = rows[-1][1] / max(rows[-1][2], 1)
if ratio >= 1:
    ok(f"final card is {ratio:.1f}x smaller than the transcript it stands for")
else:
    ok(f"final card is {1 / ratio:.1f}x LARGER than the transcript - read on")

### Read that chart sceptically

You may well have just watched the card grow *past* the transcript. That is not a
bug in the run; it is the direct consequence of a choice made two cells ago, and it
is worth sitting with.

`context.merge_card` is **additive**: the model can add a fact or correct one, but
it cannot delete. That was deliberate - it means any fact missing from the final
card was genuinely lost by summarisation rather than quietly overwritten by us, so
the probe in the next section measures something real. The cost is that nothing
ever leaves, and a card that never forgets is not a compaction card. It is a log.

**Both halves of that trade are the lesson.** A card that forgets nothing has
perfect fidelity and no compression. A card that forgets aggressively has excellent
compression and unknown fidelity - and "unknown" is the part the original section
never measured, because it only ever plotted the size.

Real systems bound the card: keep at most N facts, evict by age or by how recently
a fact was referenced, and *re-run the probe after every eviction policy change*.
Notebook 04 does exactly that as a scored eval. Here we hold the policy still at
one extreme so the measurement below is unambiguous.

Either way, the chart alone could never tell you which regime you are in. So we go
and ask.

## 4 · The probe: is it still in there?

Turn 4 planted a reason. Thirty-six turns later, we ask the question whose answer
*requires* that reason - with nothing to answer from but the card.

Two levels of scoring, because they can disagree in an interesting way:

- **Structural**: `context.card_fidelity` asks whether the fact id survived at all.
- **Functional**: does the model's answer actually contain the planted detail?

A fact can survive structurally and still be useless if compaction blurred it into
something too generic to answer with.

In [ ]:
PROBES = {
    "f1": ("Which pier did we flag, and what single reason put Harbor Bridge"
           " at the top of the Q3 list?", "pier 3"),
    "f2": ("Are all of this season's grades comparable with each other? If not,"
           " which inspection is the exception and why?", "2019"),
    "f3": ("Whose sign-off should a reviewer be careful about, and across which"
           " district?", "okafor"),
}

final_card = latest_card()
hits, misses, recall = context.card_fidelity(list(PROBES), final_card)
print(f"structural recall: {recall:.0%}  (kept: {hits or '-'}, lost: {misses or '-'})")

answered = []
# ✏️ TODO(3): for each probe, ask the model using ONLY the card as context,
#   and record whether the expected detail appears in its answer.
#   ... your code here ...

functional = sum(answered) / len(PROBES)
print(f"functional recall: {functional:.0%}")

# Deliberately not 100%: compaction really does lose things, and a notebook that
# only passes when nothing is ever lost would be teaching a fiction.
check(recall >= 2 / 3, f"card retained {recall:.0%} of facts planted up to 36 turns earlier")
if misses:
    print(f"lost along the way: {', '.join(misses)} - look at which card version dropped it")

### What the misses tell you

Facts survive compaction when the card has a **slot** for them - that is the whole
reason the card has a fixed schema rather than being a paragraph of prose. The
first thing lossy summarisation discards is the category with the weakest slot,
which is almost always `open_questions`: they read as incidental, and nothing in
the transcript ever refers back to them.

If a fact went missing above, find the version where it vanished
(`SELECT card_version, card_json FROM HARNESS_CARD ORDER BY card_version`) and look
at what the model was folding in at that moment. That, not the character count, is
the feedback loop that improves a compaction prompt.

## 5 · Offloading large tool results

Compaction manages the conversation. Offloading manages the *attachments*: a tool
returns 40KB of JSON, and putting it in the context window would evict everything
else. So we store the payload, hand the model a **reference**, and let it decide.

The original section stopped at "the bytes are recoverable", which is a property of
the storage layer, not of the agent. The interesting question is behavioural: shown
a reference instead of data, does the model *notice*, *fetch*, and *use* it? We are
going to ask it something it cannot possibly answer from the stub, and check.

In [ ]:
ddl("""CREATE TABLE HARNESS_BLOB (
         BLOB_ID      VARCHAR2(12) PRIMARY KEY,
         CREATED_AT   TIMESTAMP DEFAULT SYSTIMESTAMP,
         SOURCE_TOOL  VARCHAR2(60),
         PATH         VARCHAR2(400),
         PAYLOAD      CLOB,
         BYTES        NUMBER)""")


def store_blob(source_tool, payload):
    blob_id = context.new_blob_id()
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO HARNESS_BLOB (blob_id, source_tool, path, payload, bytes)"
            " VALUES (:1, :2, :3, :4, :5)",
            (blob_id, source_tool, context.blob_path(blob_id), payload, len(payload)),
        )
    conn.commit()
    return blob_id


def fetch_blob(blob_id):
    with conn.cursor() as cur:
        cur.execute("SELECT payload FROM HARNESS_BLOB WHERE blob_id = :1", (blob_id,))
        row = cur.fetchone()
    if not row:
        return None
    return row[0].read() if hasattr(row[0], "read") else row[0]

ok("blob store ready")

In [ ]:
# The busiest asset gives us an export big enough to be worth offloading.
with conn.cursor() as cur:
    cur.execute(
        "SELECT asset_id FROM CITY_INSPECTION_FINDING"
        " GROUP BY asset_id ORDER BY COUNT(*) DESC, asset_id FETCH FIRST 1 ROWS ONLY"
    )
    (TARGET_ASSET,) = cur.fetchone()

with conn.cursor() as cur:
    cur.execute(
        "SELECT finding_id, severity, category, DBMS_LOB.SUBSTR(description, 600, 1), days_ago"
        "  FROM CITY_INSPECTION_FINDING WHERE asset_id = :1 ORDER BY days_ago",
        (TARGET_ASSET,),
    )
    EXPORT_ROWS = [
        {"finding_id": r[0], "severity": r[1], "category": r[2],
         "description": r[3], "days_ago": r[4]}
        for r in cur.fetchall()
    ]

# Ground truth, computed in Python - the model has to match this, not be graded by it.
# A *lookup*, deliberately, not a count. Asking for a tally over 36 rows tests the
# model's arithmetic, and any number it produces might appear in the answer for
# unrelated reasons (list numbering, for one). An opaque finding_id cannot appear
# unless the payload was actually read - which is the only thing we want to prove.
# The sort key breaks ties explicitly so the question has exactly one right answer.
OLDEST = max(EXPORT_ROWS, key=lambda r: (r["days_ago"], r["finding_id"]))
EXPORT_JSON = json.dumps(EXPORT_ROWS, default=str)

check(context.offload_decision(len(EXPORT_JSON)) == "offload",
      f"{TARGET_ASSET} export is {len(EXPORT_JSON)} chars - too big to inline")
print(f"ground truth: oldest finding is {OLDEST['finding_id']} "
      f"({OLDEST['days_ago']} days ago, {OLDEST['severity']} {OLDEST['category']})")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

FETCH_LOG = []   # every blob_id the model asked for, in order


@tool
def bulk_inspection_export(asset_id: str) -> str:
    """Export every inspection finding for an asset. Large results are offloaded
    and returned as a harness://blob/<id> reference instead of inline data."""
    payload = EXPORT_JSON
    if context.offload_decision(len(payload)) == "inline":
        return payload
    blob_id = store_blob("bulk_inspection_export", payload)
    return (f"[result offloaded: {context.blob_reference(blob_id)} - "
            f"{len(EXPORT_ROWS)} findings, {len(payload)} bytes]")


@tool
def fetch_offloaded(blob_id: str) -> str:
    """Retrieve the full payload of a previously offloaded result by its blob id."""
    # ✏️ TODO(4): record the fetch in FETCH_LOG and return the payload.
    #   Return a clear error string if the id is unknown - the model should be able
    #   to recover from a bad id, not crash the run.
    #   ... your code here ...


OFFLOAD_TOOLS = {t.name: t for t in [bulk_inspection_export, fetch_offloaded]}
ok(f"{len(OFFLOAD_TOOLS)} tools defined - one offloads, one redeems the reference")

In [ ]:
OFFLOAD_SYSTEM = (
    "You are the CityOps inspection copilot. You act through tools.\n"
    "Some tool results are too large to return inline; they come back as a\n"
    "harness://blob/<id> reference. A reference is not an answer - if you need what\n"
    "is inside it, call fetch_offloaded with that id.\n"
    "Answer only from data you have actually seen. Be concise: two sentences."
)

QUESTION = (f"Export all inspection findings for {TARGET_ASSET}. Which finding_id has"
            f" the largest days_ago - if several tie, the one whose finding_id sorts"
            f" last - and what are its severity and category?")

model = CHAT.bind_tools(list(OFFLOAD_TOOLS.values()))
msgs = [SystemMessage(content=OFFLOAD_SYSTEM), HumanMessage(content=QUESTION)]
tool_outputs = []
resp = None

for _ in range(6):
    resp = model.invoke(msgs, config=CFG)
    msgs.append(resp)
    if not getattr(resp, "tool_calls", None):
        break
    for tc in resp.tool_calls:
        try:
            result = str(OFFLOAD_TOOLS[tc["name"]].invoke(tc["args"]))
        except Exception as e:
            result = f"ERROR: {type(e).__name__}: {e}"
        tool_outputs.append((tc["name"], result))
        print(f"  -> {tc['name']}({tc['args']}) = {result[:90]}")
        msgs.append(ToolMessage(content=result, tool_call_id=tc["id"]))

answer = resp.content if resp is not None else ""
if isinstance(answer, list):
    answer = "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in answer)
print("\nANSWER:", answer.strip()[:400])

In [ ]:
# Three assertions, and only the third one is about the agent doing its job.
stub = next((out for name, out in tool_outputs if name == "bulk_inspection_export"), "")
offered = context.parse_blob_references(stub)
check(bool(offered), "the export tool handed back a blob reference, not 40KB of JSON")

# (a) the model derived the id from the reference it was shown - it did not guess.
check(bool(FETCH_LOG) and FETCH_LOG[0] in offered,
      f"the model noticed the reference and fetched it ({FETCH_LOG[:1]})")

# (b) the payload actually reached the context window.
fetched = next((out for name, out in tool_outputs if name == "fetch_offloaded"), "")
check(EXPORT_ROWS[0]["finding_id"] in fetched,
      "the fetched payload entered the conversation")

# (c) and it was *used*. This is the check the original section never made.
#     The id is opaque: it appears here only if the model read the payload.
check(OLDEST["finding_id"] in answer,
      f"the answer names {OLDEST['finding_id']} - a value only present in the payload")

If any of those failed, read the trace above rather than re-running until it
passes. A model that ignores the reference and answers from the stub is telling you
something true about your prompt: the system message has to say plainly that a
reference is not an answer. "Bytes are recoverable" would have scored this run
green either way - which is exactly why it was the wrong measurement.

## 6 · Offloaded blobs must never be promoted

This is the interaction the review flagged as untested. Notebook 02 promotes
curated notes into long-term memory. Offloading writes large, machine-generated
dumps to disk-like paths. If those two features meet without a rule between them,
a 40KB JSON export gets embedded, chunked and remembered forever as though a human
had written it down on purpose.

So we put the blob paths in front of notebook 02's real gate and watch what happens.

In [ ]:
with conn.cursor() as cur:
    cur.execute("DELETE FROM HARNESS_SCRATCH WHERE path LIKE '/tool_out/blob/%'")
    cur.execute("SELECT blob_id, path FROM HARNESS_BLOB")
    blob_rows = cur.fetchall()
    for blob_id, path in blob_rows:
        cur.execute(
            "INSERT INTO HARNESS_SCRATCH (path, content, status) VALUES (:1, :2, 'N')",
            (path, f"offloaded export {blob_id}"),
        )
    # ...and one genuine curated note, so we can tell a rule from a blanket veto.
    cur.execute("DELETE FROM HARNESS_SCRATCH WHERE path = :1", ("/inbox/" + TARGET_ASSET + "/season.md",))
    cur.execute(
        "INSERT INTO HARNESS_SCRATCH (path, content, status) VALUES (:1, :2, 'N')",
        (f"/inbox/{TARGET_ASSET}/season.md",
         f"Season summary for {TARGET_ASSET}: {len(EXPORT_ROWS)} findings on file; "
         f"the oldest still open is {OLDEST['finding_id']}."),
    )
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT path FROM HARNESS_SCRATCH WHERE status = 'N'")
    candidates = [r[0] for r in cur.fetchall()]
print(f"{len(candidates)} notes awaiting triage")

In [ ]:
def promotable(path):
    """Notebook 02's opt-in gate, plus an explicit bar on offloaded payloads."""
    # ✏️ TODO(5): a note is promotable only if it is inside the opt-in area AND
    #   is not an offloaded blob. Use promote.triage_allowed and
    #   context.is_offloaded_blob.
    #   ... your code here ...


verdicts = {p: promotable(p) for p in candidates}
with conn.cursor() as cur:
    for path, allowed in verdicts.items():
        cur.execute("UPDATE HARNESS_SCRATCH SET status = :1 WHERE path = :2",
                    ("S" if allowed else "D", path))
conn.commit()

for path, allowed in sorted(verdicts.items()):
    print(f"  {'PROMOTE' if allowed else 'discard'}  {path}")

blobs = [p for p in verdicts if context.is_offloaded_blob(p)]
check(bool(blobs) and not any(verdicts[p] for p in blobs),
      f"all {len(blobs)} offloaded blob(s) refused promotion")
check(any(allowed for path, allowed in verdicts.items() if not context.is_offloaded_blob(path)),
      "a genuinely curated /inbox note still promotes - a rule, not a blanket veto")

### Belt *and* braces, on purpose

`/tool_out/` already sits outside notebook 02's `/inbox/` opt-in prefix, so today
`triage_allowed` alone would have refused these paths. The second condition looks
redundant - and that is precisely the argument for writing it down.

The opt-in prefix is a *product* decision that will change (someone will widen it
to `/inbox/` plus `/reports/`, or drop the prefix scheme entirely). The rule that
machine-generated offload payloads are not memories is an *architectural* one that
should survive that change. Encoding it as an accident of string prefixes means it
silently stops holding the day someone edits an unrelated constant.

## 7 · State check

In [ ]:
for desc, passed in verify(conn, "03_context_engineering"):
    check(passed, desc)
ok("context engineering complete - continue to 04_evals")

### What notebook 04 does with this

Two of the five evals are this notebook, run properly:

- **Card fidelity over a long horizon** generalises §4. One season and three
  planted facts is an anecdote; the eval runs multi-topic, adversarially spaced
  sessions and reports drift as a distribution, not a single percentage.
- **Cost per correct answer** puts §3 and §5 on the same axis as the alternatives -
  card vs. full transcript vs. stateless - with repeated trials and variance, which
  is what the original 8.4 lacked at n=1.

The habit to carry forward is the one this notebook is built around: every time you
make the context smaller, you have made a claim about what is still in there. Write
the probe that tests the claim, and let it fail sometimes.